In [124]:
# import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.utils.class_weight import compute_class_weight
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
import nltk
import re
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.stem import PorterStemmer, WordNetLemmatizer
from langdetect import detect
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')
from scikeras.wrappers import KerasClassifier
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense,Dropout,Input
from lime.lime_text import LimeTextExplainer
#from imblearn.over_sampling import SMOTE




[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [95]:
# load the dataset
df = pd.read_csv('judge-1377884607_tweet_product_company.csv', encoding='latin1')
df.head()

,tweet_text,emotion_in_tweet_is_directed_at,is_there_an_emotion_directed_at_a_brand_or_product
0,.@wesley83 I have a 3G iPhone. After 3 hrs twe...,iPhone,Negative emotion
1,@jessedee Know about @fludapp ? Awesome iPad/i...,iPad or iPhone App,Positive emotion
2,@swonderlin Can not wait for #iPad 2 also. The...,iPad,Positive emotion
3,@sxsw I hope this year's festival isn't as cra...,iPad or iPhone App,Negative emotion
4,@sxtxstate great stuff on Fri #SXSW: Marissa M...,Google,Positive emotion


In [96]:
# check info
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9093 entries, 0 to 9092
Data columns (total 3 columns):
 #   Column                                              Non-Null Count  Dtype 
---  ------                                              --------------  ----- 
 0   tweet_text                                          9092 non-null   object
 1   emotion_in_tweet_is_directed_at                     3291 non-null   object
 2   is_there_an_emotion_directed_at_a_brand_or_product  9093 non-null   object
dtypes: object(3)
memory usage: 213.2+ KB


In [97]:
# check for duplicates
df.duplicated().sum()

22

In [98]:
# check for missing values
df.isna().sum()

tweet_text                                               1
emotion_in_tweet_is_directed_at                       5802
is_there_an_emotion_directed_at_a_brand_or_product       0
dtype: int64

In [99]:
# drop duplicates
df.drop_duplicates()

,tweet_text,emotion_in_tweet_is_directed_at,is_there_an_emotion_directed_at_a_brand_or_product
0,.@wesley83 I have a 3G iPhone. After 3 hrs twe...,iPhone,Negative emotion
1,@jessedee Know about @fludapp ? Awesome iPad/i...,iPad or iPhone App,Positive emotion
2,@swonderlin Can not wait for #iPad 2 also. The...,iPad,Positive emotion
3,@sxsw I hope this year's festival isn't as cra...,iPad or iPhone App,Negative emotion
4,@sxtxstate great stuff on Fri #SXSW: Marissa M...,Google,Positive emotion
...,...,...,...
9088,Ipad everywhere. #SXSW {link},iPad,Positive emotion
9089,"Wave, buzz... RT @mention We interrupt your re...",NaN,No emotion toward brand or product
9090,"Google's Zeiger, a physician never reported po...",NaN,No emotion toward brand or product
9091,Some Verizon iPhone customers complained their...,NaN,No emotion toward brand or product


In [100]:
# drop the emotion_in_tweet_is_directed_at as it contains much missing values
df.drop(columns='emotion_in_tweet_is_directed_at', inplace=True)

In [101]:
# rename columns
df = df.rename(columns={'tweet_text':'text','is_there_an_emotion_directed_at_a_brand_or_product': 'sentiment'})

In [102]:
# create another dataframe caaled df_sentiment
df_sentiment = df.copy()
df_sentiment

,text,sentiment
0,.@wesley83 I have a 3G iPhone. After 3 hrs twe...,Negative emotion
1,@jessedee Know about @fludapp ? Awesome iPad/i...,Positive emotion
2,@swonderlin Can not wait for #iPad 2 also. The...,Positive emotion
3,@sxsw I hope this year's festival isn't as cra...,Negative emotion
4,@sxtxstate great stuff on Fri #SXSW: Marissa M...,Positive emotion
...,...,...
9088,Ipad everywhere. #SXSW {link},Positive emotion
9089,"Wave, buzz... RT @mention We interrupt your re...",No emotion toward brand or product
9090,"Google's Zeiger, a physician never reported po...",No emotion toward brand or product
9091,Some Verizon iPhone customers complained their...,No emotion toward brand or product


In [103]:
# check the value counts of sentiment
df_sentiment['sentiment'].value_counts()

sentiment
No emotion toward brand or product    5389
Positive emotion                      2978
Negative emotion                       570
I can't tell                           156
Name: count, dtype: int64

In [104]:
# remove whitespaces and convert the text column to string
df_sentiment['text'] = df_sentiment['text'].fillna('').astype(str)

# do feature engineering to find out the number of words, characters and sentences
df_sentiment['words'] = df_sentiment['text'].apply(len)
df_sentiment['chars'] = df_sentiment['text'].apply(lambda text: len(nltk.word_tokenize(text)))
df_sentiment['sent'] = df_sentiment['text'].apply(lambda text: len(nltk.sent_tokenize(text)))

df_sentiment

,text,sentiment,words,chars,sent
0,.@wesley83 I have a 3G iPhone. After 3 hrs twe...,Negative emotion,127,32,5
1,@jessedee Know about @fludapp ? Awesome iPad/i...,Positive emotion,139,29,3
2,@swonderlin Can not wait for #iPad 2 also. The...,Positive emotion,79,20,2
3,@sxsw I hope this year's festival isn't as cra...,Negative emotion,82,21,2
4,@sxtxstate great stuff on Fri #SXSW: Marissa M...,Positive emotion,131,29,1
...,...,...,...,...,...
9088,Ipad everywhere. #SXSW {link},Positive emotion,29,8,2
9089,"Wave, buzz... RT @mention We interrupt your re...",No emotion toward brand or product,125,26,1
9090,"Google's Zeiger, a physician never reported po...",No emotion toward brand or product,145,32,3
9091,Some Verizon iPhone customers complained their...,No emotion toward brand or product,140,26,2


In [105]:
# check for descriptive statistics
df_sentiment.describe()

,words,chars,sent
count,9093.000000,9093.000000,9093.000000
mean,104.950731,24.412295,1.880018
std,27.208419,6.505483,0.943527
min,0.000000,0.000000,0.000000
25%,86.000000,20.000000,1.000000
50%,109.000000,25.000000,2.000000
75%,126.000000,29.000000,2.000000
max,178.000000,49.000000,7.000000


In [106]:
# label encode the target
le = LabelEncoder()
df_sentiment['sentiment'] = le.fit_transform(df_sentiment['sentiment'])
df_sentiment['sentiment'] 


0       1
1       3
2       3
3       1
4       3
       ..
9088    3
9089    2
9090    2
9091    2
9092    2
Name: sentiment, Length: 9093, dtype: int32

In [ ]:
# create an oop function to do data preprocessing

class SentimentPreprocessor:
    def __init__(self, text_column, target_column):
        self.text_column = text_column
        self.target_column = target_column
        self.vectorizer = TfidfVectorizer()
        self.stop_words = set(stopwords.words('english'))
        self.lemmatizer = WordNetLemmatizer()

    def _is_english(self, text):
        try:
            return detect(text) == 'en'
        except:
            return False

    def _remove_emojis(self, text):
        emoji_pattern = re.compile("["  
                                   u"\U0001F600-\U0001F64F"
                                   u"\U0001F300-\U0001F5FF"
                                   u"\U0001F680-\U0001F6FF"
                                   u"\U0001F1E0-\U0001F1FF"
                                   "]+", flags=re.UNICODE)
        return emoji_pattern.sub(r'', text)

    def _clean_text(self, text):
        text = self._remove_emojis(text)
        text = re.sub(r'\d+', '', text)
        text = text.lower()
        text = text.translate(str.maketrans('', '', string.punctuation))
        return text

    def _tokenize(self, text):
        return word_tokenize(text)

    def _remove_stopwords(self, tokens):
        return [word for word in tokens if word.isalpha() and word not in self.stop_words]

    def _lemmatize_tokens(self, tokens):
        return [self.lemmatizer.lemmatize(token) for token in tokens]

    def preprocess(self, df_sentiment):
        print("1. Removing non-English rows")
        df_sentiment = df_sentiment[df_sentiment[self.text_column].astype(str).apply(self._is_english)].copy()

        print("2. Cleaning raw text")
        df_sentiment["clean_text"] = df_sentiment[self.text_column].astype(str).apply(self._clean_text)

        print("3. Tokenizing text")
        df_sentiment["tokenized_text"] = df_sentiment["clean_text"].apply(self._tokenize)

        print("4. Removing stopwords")
        df_sentiment["stopwords_removed"] = df_sentiment["tokenized_text"].apply(self._remove_stopwords)

        print("5. Lemmatizing tokens")
        df_sentiment["lemmatized_text"] = df_sentiment["stopwords_removed"].apply(self._lemmatize_tokens)

        print("6. Joining lemmatized tokens for final clean text")
        df_sentiment["final_clean_text"] = df_sentiment["lemmatized_text"].apply(lambda tokens: " ".join(tokens))

        print("7. Vectorizing 'final_clean_text' using TF-IDF")
        X_tfidf = self.vectorizer.fit_transform(df_sentiment["final_clean_text"])
        y = df_sentiment[self.target_column].values

        return df_sentiment, X_tfidf, y

    def split_and_process(self, df_sentiment, test_size=0.2):
        df_cleaned, X, y = self.preprocess(df_sentiment)
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=42)
        return df_cleaned, X_train, X_test, y_train, y_test




In [108]:
# apply the class on the data

processor = SentimentPreprocessor(text_column='text', target_column='sentiment')
df_cleaned, X_train, X_test, y_train, y_test = processor.split_and_process(df_sentiment)

# Print all relevant columns with intermediate steps
print(df_cleaned[['text', 'clean_text', 'tokenized_text', 'stopwords_removed', 'lemmatized_text', 'final_clean_text']].head())


1. Removing non-English rows
2. Cleaning raw text
3. Tokenizing text
4. Removing stopwords
5. Lemmatizing tokens
6. Joining lemmatized tokens for final clean text
7. Vectorizing 'final_clean_text' using TF-IDF
                                                text  \
0  .@wesley83 I have a 3G iPhone. After 3 hrs twe...   
1  @jessedee Know about @fludapp ? Awesome iPad/i...   
2  @swonderlin Can not wait for #iPad 2 also. The...   
3  @sxsw I hope this year's festival isn't as cra...   
4  @sxtxstate great stuff on Fri #SXSW: Marissa M...   

                                          clean_text  \
0  wesley i have a g iphone after  hrs tweeting a...   
1  jessedee know about fludapp  awesome ipadiphon...   
2  swonderlin can not wait for ipad  also they sh...   
3  sxsw i hope this years festival isnt as crashy...   
4  sxtxstate great stuff on fri sxsw marissa maye...   

                                      tokenized_text  \
0  [wesley, i, have, a, g, iphone, after, hrs, tw...   
1  [

In [127]:
# 1. Neural network definition using Input layer
def create_nn():
    model = Sequential([
        Input(shape=(X_train.shape[1],)),
        Dense(128, activation='relu'),
        Dropout(0.3),
        Dense(64, activation='relu'),
        Dropout(0.2),
        Dense(len(le.classes_), activation='softmax')
    ])
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

# 2. Define pipelines for each model
pipelines = {
    "Decision Tree": Pipeline([
        ('model', DecisionTreeClassifier())
    ]),
    "Random Forest": Pipeline([
        ('model', RandomForestClassifier())
    ]),
    "XGBoost": Pipeline([
        ('model', XGBClassifier())
    ]),
    "Neural Network": Pipeline([
        ('scaler', StandardScaler(with_mean=False)),
        ('model', KerasClassifier(model=create_nn, epochs=5, batch_size=32, verbose=0))
    ])
}

# 3. Cross-validation
print("Model Performance (3-fold CV with pipeline):")
cv_scores = {}
for name, pipe in pipelines.items():
    score = cross_val_score(pipe, X_train, y_train, cv=3, scoring='accuracy')
    cv_scores[name] = score.mean()
    print(f"{name}: {score.mean():.2f} accuracy")

# 4. Best model
best_model_name = max(cv_scores, key=cv_scores.get)
print(f"Best Model: {best_model_name}")

# 5. Fit final pipeline and test
final_pipe = pipelines[best_model_name]
final_pipe.fit(X_train, y_train)

print("Sample Predictions:")
for i in range(3):
    text = df_cleaned.iloc[i]['final_clean_text']
    true_label = df_cleaned.iloc[i]['sentiment']
    pred = final_pipe.predict(X_train[i].reshape(1, -1))
    pred_label = le.inverse_transform(pred)[0]

    print(f"Text: {text[:50]}...")
    print(f"True: {true_label}")
    print(f"Predicted: {pred_label}")

Model Performance (3-fold CV with pipeline):
Decision Tree: 0.59 accuracy
Random Forest: 0.65 accuracy
XGBoost: 0.66 accuracy
Neural Network: 0.64 accuracy
Best Model: XGBoost
Sample Predictions:
Text: wesley g iphone hr tweeting riseaustin dead need u...
True: 1
Predicted: No emotion toward brand or product
Text: jessedee know fludapp awesome ipadiphone app youll...
True: 3
Predicted: Positive emotion
Text: swonderlin wait ipad also sale sxsw...
True: 3
Predicted: No emotion toward brand or product


In [ ]:
# 1. Encode class weights
class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(y_train), y=y_train)
class_weight_dict = dict(enumerate(class_weights))

# 2. Define Neural Network
def create_nn():
    model = Sequential([
        Input(shape=(X_train.shape[1],)),
        Dense(64, activation='relu'),
        Dense(len(le.classes_), activation='softmax')
    ])
    model.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    return model

# 3. Create pipelines
pipelines = {
    "Decision Tree": Pipeline([
        ('model', DecisionTreeClassifier(class_weight='balanced'))
    ]),
    "Random Forest": Pipeline([
        ('model', RandomForestClassifier(class_weight='balanced'))
    ]),
    "XGBoost": Pipeline([
        ('model', XGBClassifier(scale_pos_weight=class_weights.max()))
    ]),
    "Neural Net": Pipeline([
        ('scaler', StandardScaler()),
        ('model', KerasClassifier(model=create_nn,
                                  epochs=5,
                                  batch_size=32,
                                  verbose=0))
    ])
}

# 4. Cross-validation comparison
print("Model Performance:")
cv_scores = {}
for name, pipe in pipelines.items():
    scores = cross_val_score(pipe, X, y_encoded, cv=3, scoring='accuracy')
    cv_scores[name] = scores.mean()
    print(f"{name}: {scores.mean():.2f} accuracy")

# 5. Train final model (Neural Net example with weight support)
nn_model = KerasClassifier(model=create_nn, epochs=5, batch_size=32, verbose=1)
nn_model.fit(X_train, y_tra, class_weight=class_weight_dict)
# 6. Prediction on sample input
sample_text = df_cleaned.iloc[0]['final_clean_text']
pred = le.inverse_transform(nn_model.predict(X[0].reshape(1, -1)))[0]
print(f"Sample Prediction:")
print(f"Text: {sample_text[:50]}...")
print(f"Predicted: {pred}")
print(f"Actual: {df_cleaned.iloc[0]['sentiment']}")
